# 🎙️ Narrador documental en español · Qwen3-TTS en Google Colab

Gratis, con la GPU de Google. **Antes de empezar:** menú **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4 → Guardar**.

Ejecuta las celdas **en orden** con el botón ▶️ de cada una:

1. **Instalar** (2–3 min, solo la primera vez en cada sesión)
2. **Guardar en Google Drive** (recomendado: así la voz elegida no se pierde)
3. **Elegir la voz**: escúchala y cambia la *semilla* hasta que te guste
4. **Pegar el guion y narrar**: descarga el WAV y los subtítulos SRT
5. *(Opcional)* **Pruebas de pronunciación**

Modelos: Qwen3-TTS 1.7B (Apache-2.0, uso comercial permitido). Pon «Narración generada con IA» en la descripción de tus videos.

## 1 · Instalar

In [ ]:
#@title ▶️ Instalar (ejecuta y espera al ✅)
import subprocess, sys, torch
if not torch.cuda.is_available():
    raise SystemExit("❌ No hay GPU. Menú Entorno de ejecución → Cambiar tipo de entorno → GPU T4, y vuelve a ejecutar.")
print("GPU:", torch.cuda.get_device_name(0))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "qwen-tts", "num2words", "pyloudnorm", "jiwer"], check=True)
print("✅ Instalado. Si más abajo aparece un error de 'transformers', usa Entorno de ejecución → Reiniciar sesión y ejecuta desde la celda 2.")

## 2 · Guardar en Google Drive (recomendado)

In [ ]:
#@title ▶️ Conectar Drive (las voces y narraciones se guardan en MyDrive/Narrador)
usar_drive = True #@param {type:"boolean"}
from pathlib import Path
if usar_drive:
    from google.colab import drive
    drive.mount("/content/drive")
    CARPETA = Path("/content/drive/MyDrive/Narrador")
else:
    CARPETA = Path("/content/Narrador")
(CARPETA / "voces").mkdir(parents=True, exist_ok=True)
(CARPETA / "narraciones").mkdir(parents=True, exist_ok=True)
print("📁 Carpeta de trabajo:", CARPETA)

### Motor (no hace falta tocar nada; solo ejecútala)

In [ ]:
%%writefile /content/texto.py
"""Preparación del texto en español: normalización para TTS y división en segmentos.

Qwen3-TTS lee mejor el texto "como se pronuncia": cifras, siglas y símbolos
escritos en palabras. Todo lo que no esté en el léxico se deja igual.
"""
from __future__ import annotations

import json
import re
from dataclasses import dataclass
from pathlib import Path

try:
    from num2words import num2words
except ImportError:  # num2words es opcional; sin él las cifras se dejan como están
    num2words = None

LEXICO_POR_DEFECTO = Path("/content/lexico.json")


# ---------------------------------------------------------------- léxico
def cargar_lexico(ruta: str | Path | None = None) -> dict[str, str]:
    ruta = Path(ruta) if ruta else LEXICO_POR_DEFECTO
    if not ruta.exists():
        return {}
    datos = json.loads(ruta.read_text(encoding="utf-8"))
    return {k: v for k, v in datos.items() if not k.startswith("_")}


def aplicar_lexico(texto: str, lexico: dict[str, str]) -> str:
    # Las entradas más largas primero, para que "US$" no lo pise "US".
    for termino in sorted(lexico, key=len, reverse=True):
        patron = r"(?<![\wÁÉÍÓÚÑáéíóúñ])" + re.escape(termino) + r"(?![\wÁÉÍÓÚÑáéíóúñ])"
        texto = re.sub(patron, lexico[termino], texto)
    return texto


# ---------------------------------------------------------------- números
FEMENINOS = {
    "personas", "víctimas", "muertes", "escuelas", "tropas", "toneladas", "familias", "horas", "semanas",
    "mujeres", "niñas", "viviendas", "comunidades", "organizaciones", "elecciones", "vidas", "casas",
    "hectáreas", "ciudades", "deportaciones", "expulsiones", "camas", "plazas", "fuerzas", "bandas",
    "pandillas", "persona", "víctima", "escuela", "vez",
}


def _palabras(n: int | float, siguiente: str = "") -> str:
    if num2words is None:
        return str(n)
    t = num2words(n, lang="es")
    if siguiente.lower() in FEMENINOS:
        t = re.sub(r"ientos\b", "ientas", t)
        t = re.sub(r"\buno$", "una", t).replace("veintiuno", "veintiuna")
        t = re.sub(r"\buno\b(?= mil)", "un", t)
    elif siguiente and re.match(r"[a-záéíóúñ]", siguiente.lower()):
        t = re.sub(r"uno$", "ún" if t.endswith("veintiuno") else "un", t).replace("veintiún", "veintiún")
    return t


def _entero(cadena: str) -> int:
    return int(cadena.replace(".", "").replace("\u202f", "").replace(" ", ""))


_SIG = r"(?=(?:\s+(?:de\s+)?([a-záéíóúñ]+))?)"   # captura la palabra siguiente (para concordar género)


def normalizar_numeros(texto: str) -> str:
    if num2words is None:
        return texto
    # Rangos "80–90 %" -> "80 a 90 %"
    texto = re.sub(r"(\d+(?:,\d+)?)\s*[–-]\s*(\d+(?:,\d+)?)\s*%", r"\1 a \2 %", texto)
    # Porcentajes
    texto = re.sub(r"(\d+(?:,\d+)?)\s*%", r"\1 por ciento", texto)
    # Moneda: "US$ 880 millones" -> "880 millones de dólares"; "US$ 50" -> "50 dólares"
    texto = re.sub(r"US\$\s*(\d[\d.]*(?:,\d+)?)(\s+(?:mil\s+)?millones|\s+mil)?",
                   lambda m: f"{m.group(1)}{m.group(2) or ''} {'de ' if m.group(2) and 'millones' in m.group(2) else ''}dólares", texto)
    # Decimales con coma: "1,5" -> "uno coma cinco"
    def dec(m):
        ent, frac = m.group(1), m.group(2)
        frac_txt = _palabras(int(frac)) if not frac.startswith("0") else " ".join(_palabras(int(c)) for c in frac)
        return f"{_palabras(_entero(ent))} coma {frac_txt}"
    texto = re.sub(r"(?<![\d.])(\d{1,3}(?:\.\d{3})*|\d+),(\d+)(?![\d.]|,\d)", dec, texto)
    # Ordinales "1.º", "3.ª"
    texto = re.sub(r"(\d+)\.?\s*º(?=\s+[a-záéíóúñ])", lambda m: re.sub(r"(prim|terc)ero$", r"\1er", num2words(int(m.group(1)), lang="es", to="ordinal")), texto)
    texto = re.sub(r"(\d+)\.?\s*º", lambda m: num2words(int(m.group(1)), lang="es", to="ordinal"), texto)
    texto = re.sub(r"(\d+)\.?\s*ª", lambda m: num2words(int(m.group(1)), lang="es", to="ordinal")[:-1] + "a", texto)
    # Miles con punto o espacio fino: "4.217 muertes" -> "cuatro mil doscientas diecisiete muertes"
    texto = re.sub(r"(?<![\d,])(\d{1,3}(?:[.\u202f]\d{3})+)(?!\d|,\d)" + _SIG,
                   lambda m: _palabras(_entero(m.group(1)), m.group(2) or ""), texto)
    # Resto de enteros (años incluidos)
    texto = re.sub(r"(?<!\w)(?<!\d,)(\d+)(?!\w|,\d)" + _SIG,
                   lambda m: _palabras(int(m.group(1)), m.group(2) or ""), texto)
    return texto


def normalizar(texto: str, lexico: dict[str, str] | None = None) -> str:
    texto = texto.replace(" ", " ")
    texto = re.sub(r"\[\d+\]", "", texto)            # quita referencias tipo [12]
    texto = re.sub(r"\[[A-ZÁÉÍÓÚ ]+\]", "", texto)    # quita marcas tipo [VERIFICAR]
    # Rangos de años/cifras "1957–1986" -> "1957 a 1986" (los % se tratan después)
    texto = re.sub(r"(\d)\s*[–—-]\s*(\d[\d.,]*)(?!\s*%)", r"\1 a \2", texto)
    # Rayas de inciso -> pausa breve
    texto = re.sub(r"\s*[—–]\s*", ", ", texto)
    if lexico:
        texto = aplicar_lexico(texto, lexico)
    texto = normalizar_numeros(texto)
    texto = re.sub(r"\s+([.,;:!?…])", r"\1", texto)
    texto = re.sub(r",\s*,", ",", texto)
    texto = re.sub(r"[ \t]+", " ", texto)
    return texto.strip()


# ---------------------------------------------------------------- división
@dataclass
class Segmento:
    texto: str
    parrafo: int          # índice del párrafo (para pausas más largas entre párrafos)
    final_de_parrafo: bool


def dividir(texto: str, max_caracteres: int = 280) -> list[Segmento]:
    """Divide en párrafos (líneas en blanco) y, si un párrafo es largo,
    en grupos de frases completas de hasta `max_caracteres`.

    Segmentos cortos dan una voz más estable: el modelo pierde el hilo
    (repeticiones, cambios de ritmo) con entradas muy largas.
    """
    parrafos = [p.strip() for p in re.split(r"\n\s*\n", texto) if p.strip()]
    segmentos: list[Segmento] = []
    for i, p in enumerate(parrafos):
        p = re.sub(r"\s*\n\s*", " ", p)
        frases = re.split(r"(?<=[.!?…;:])\s+(?=[¿¡\"“A-ZÁÉÍÓÚÑ0-9])", p)
        grupo = ""
        trozos: list[str] = []
        for f in frases:
            if len(f) > max_caracteres:  # frase larguísima: corta por comas
                for sub in re.split(r"(?<=,)\s+", f):
                    if grupo and len(grupo) + len(sub) + 1 > max_caracteres:
                        trozos.append(grupo); grupo = sub
                    else:
                        grupo = f"{grupo} {sub}".strip()
                continue
            if grupo and len(grupo) + len(f) + 1 > max_caracteres:
                trozos.append(grupo); grupo = f
            else:
                grupo = f"{grupo} {f}".strip()
        if grupo:
            trozos.append(grupo)
        for j, t in enumerate(trozos):
            segmentos.append(Segmento(texto=t, parrafo=i, final_de_parrafo=(j == len(trozos) - 1)))
    return segmentos

In [ ]:
#@title ▶️ Cargar motor y léxico de pronunciación
import json, math, re, time, gc
import numpy as np, soundfile as sf, torch
from IPython.display import Audio, display, HTML
import sys; sys.path.insert(0, "/content")
from texto import dividir, normalizar

# Léxico: a la izquierda cómo aparece en el guion, a la derecha cómo debe sonar. Edítalo aquí.
LEXICO = {
  "ONU": "Onu",
  "BINUH": "Binú",
  "OIM": "o i eme",
  "IPC": "i pe ce",
  "GSF": "ge ese efe",
  "CPT": "ce pe te",
  "CEP": "ce e pe",
  "OEA": "o e a",
  "OCHA": "Ocha",
  "UNICEF": "Unicef",
  "ACNUR": "Acnur",
  "CARICOM": "Caricom",
  "MSS": "eme ese ese",
  "UNSOH": "Unsó",
  "ACAPS": "Acaps",
  "ACLED": "Aclet",
  "MIREX": "Mírex",
  "EE. UU.": "Estados Unidos",
  "Croix-des-Bouquets": "Cruá de Buqué",
  "Viv Ansanm": "Viv Ansam",
  "Fils-Aimé": "Fis-Emé",
  "Jovenel Moïse": "Yovenel Moís",
  "Moïse": "Moís",
  "Artibonite": "Artibonít",
  "Duvalier": "Duvalié",
  "Haina": "Jaina",
  "Pedernales": "Pedernales",
  "Dajabón": "Dajabón",
  "kompa": "compá",
  "konbit": "conbít"
}
lex_dril = CARPETA / "lexico.json"
if lex_dril.exists():  # si ya guardaste un léxico en Drive, se usa ese
    LEXICO.update(json.loads(lex_dril.read_text(encoding="utf-8")))
lex_dril.write_text(json.dumps(LEXICO, ensure_ascii=False, indent=2), encoding="utf-8")

from qwen_tts import Qwen3TTSModel
DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() and "T4" not in torch.cuda.get_device_name(0) else torch.float16
_MODELOS = {}
def modelo(tipo):
    nombre = {"diseno": "Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign", "base": "Qwen/Qwen3-TTS-12Hz-1.7B-Base"}[tipo]
    if nombre not in _MODELOS:
        print("Cargando", nombre, "… (la primera vez descarga ~4 GB)")
        _MODELOS[nombre] = Qwen3TTSModel.from_pretrained(nombre, device_map="cuda:0", dtype=DTYPE, attn_implementation="sdpa")
    return _MODELOS[nombre]

def liberar(tipo):
    nombre = {"diseno": "Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign", "base": "Qwen/Qwen3-TTS-12Hz-1.7B-Base"}[tipo]
    _MODELOS.pop(nombre, None); gc.collect(); torch.cuda.empty_cache()

def recortar(a, sr, umbral_db=-45, margen=0.04):
    v = int(0.01 * sr); n = len(a) // v
    if n == 0: return a
    rms = np.sqrt(np.mean(a[: n * v].reshape(n, v) ** 2, axis=1) + 1e-12)
    act = np.where(20 * np.log10(rms) > umbral_db)[0]
    if act.size == 0: return a
    return a[max(0, act[0] * v - int(margen * sr)): min(len(a), (act[-1] + 1) * v + int(margen * sr))]

def fundidos(a, sr, ms=12):
    k = min(len(a) // 2, int(ms / 1000 * sr))
    if k > 0:
        r = np.linspace(0, 1, k); a[:k] *= r; a[-k:] *= r[::-1]
    return a

def sonoridad(a, sr, lufs=-16.0):
    import pyloudnorm as pyln
    try:
        actual = pyln.Meter(sr).integrated_loudness(a)
        if math.isfinite(actual): a = pyln.normalize.loudness(a, actual, lufs)
    except Exception:
        a = a / (np.max(np.abs(a)) or 1) * 0.7
    p = np.max(np.abs(a))
    return a / p * 0.98 if p > 0.98 else a

def srt(manifiesto):
    def ts(x):
        h, r = divmod(x, 3600); m, s = divmod(r, 60)
        return f"{int(h):02d}:{int(m):02d}:{int(s):02d},{int((s % 1) * 1000):03d}"
    return "\n".join(f"{m['n']}\n{ts(m['inicio'])} --> {ts(m['fin'])}\n{m['texto']}\n" for m in manifiesto)
print("✅ Motor listo.")

## 3 · Elegir la voz

In [ ]:
#@title ▶️ Diseñar y escuchar la voz
voz = "Narradora documental (cálida, serena)" #@param ["Narrador documental (grave, pausado)", "Narrador cercano (claro, joven)", "Narradora documental (cálida, serena)", "Narradora periodística (clara, dinámica)", "Personalizada (escribe la descripción abajo)"]
descripcion_personalizada = "" #@param {type:"string"}
expresividad = "natural" #@param ["sobria", "natural", "expresiva"]
semilla = 7 #@param {type:"integer"}

VOCES = {
  "Narrador documental (grave, pausado)": "Adult male narrator in his mid-40s, deep warm baritone, native Latin American Spanish speaker with a neutral accent, crisp diction, calm and authoritative documentary delivery, measured pace.",
  "Narrador cercano (claro, joven)": "Male narrator around 30 years old, clear bright midrange voice, native neutral Latin American Spanish, friendly and engaging explainer tone, steady pace, very clear articulation.",
  "Narradora documental (cálida, serena)": "Adult female narrator in her early 40s, warm low-mezzo voice, native Latin American Spanish speaker with a neutral accent, serene and trustworthy documentary delivery, measured pace, crisp diction.",
  "Narradora periodística (clara, dinámica)": "Female news presenter around 35, clear confident voice, native neutral Latin American Spanish, dynamic yet serious journalistic delivery, precise articulation, moderate pace.",
}
ESTILO = {"sobria": (0.6, "Calm, measured, neutral documentary narration with minimal emotion."),
          "natural": (0.75, "Natural, warm documentary narration with gentle emphasis on key facts."),
          "expresiva": (0.9, "Engaging, expressive storytelling with clear emphasis on key figures and controlled emotion.")}
TEXTO_REF = ("Bienvenidos. En este documental recorreremos la historia de un país del Caribe, "
             "sus montañas, su gente y los desafíos que enfrenta hoy, con datos verificados y mucho respeto.")

desc = descripcion_personalizada.strip() if voz.startswith("Personalizada") else VOCES[voz]
if not desc:
    raise SystemExit("Escribe una descripción personalizada (en inglés funciona mejor).")
TEMP, estilo = ESTILO[expresividad]
import unicodedata
base_nombre = unicodedata.normalize("NFKD", voz if not voz.startswith("Personalizada") else desc[:30]).encode("ascii", "ignore").decode()
slug = re.sub(r"[^a-z0-9]+", "_", base_nombre.lower()).strip("_")
REF_WAV = CARPETA / "voces" / f"{slug}__{expresividad}__s{semilla}.wav"

if REF_WAV.exists():
    print("Ya tenías esta voz guardada; la reutilizo:", REF_WAV.name)
    ref_audio, SR = sf.read(REF_WAV, dtype="float32")
else:
    torch.manual_seed(semilla); np.random.seed(semilla)
    wavs, SR = modelo("diseno").generate_voice_design(text=TEXTO_REF, language="Spanish",
                                                       instruct=f"{desc} {estilo}", temperature=0.7)
    ref_audio = sonoridad(fundidos(recortar(np.asarray(wavs[0], dtype=np.float32), SR), SR), SR)
    sf.write(REF_WAV, ref_audio, SR, subtype="PCM_16")
    REF_WAV.with_suffix(".json").write_text(json.dumps({"voz": voz, "descripcion": desc, "expresividad": expresividad,
                                                        "semilla": semilla}, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"🔊 {voz} · semilla {semilla} · {expresividad}")
display(Audio(ref_audio, rate=SR))
print("¿No te convence? Cambia la semilla (por ejemplo 8, 9, 10…) y vuelve a ejecutar. Apunta la que te guste.")

## 4 · Pegar el guion y narrar

In [ ]:
#@title ✍️ Pega tu guion entre las comillas triples (separa los párrafos con una línea en blanco)
GUION = """
¿Y quién gobierna Haití hoy? En 2024 se creó un Consejo Presidencial de Transición, el CPT, con la tarea de pacificar el país y organizar elecciones. Pero su mandato expiró el 7 de febrero de 2026 sin que se lograra celebrar los comicios.

Desde entonces, el primer ministro Alix Didier Fils-Aimé quedó al frente del Poder Ejecutivo sin presidente ni Parlamento en funciones.
"""

In [ ]:
#@title ▶️ Narrar y descargar
nombre_archivo = "narracion" #@param {type:"string"}
solo_vista_previa = False #@param {type:"boolean"}
pausa_entre_frases = 0.3 #@param {type:"number"}
pausa_entre_parrafos = 0.8 #@param {type:"number"}
ver_texto_leido = False #@param {type:"boolean"}

if "REF_WAV" not in globals():
    raise SystemExit("Primero ejecuta la celda 3 (elegir la voz).")
liberar("diseno")
base = modelo("base")
prompt = base.create_voice_clone_prompt(ref_audio=str(REF_WAV), ref_text=TEXTO_REF)

segmentos = dividir(GUION, 280)
if solo_vista_previa: segmentos = segmentos[:1]
if not segmentos: raise SystemExit("El guion está vacío.")

def generar(texto, s):
    torch.manual_seed(s); np.random.seed(s)
    w, sr = base.generate_voice_clone(text=texto, language="Spanish", voice_clone_prompt=prompt, temperature=TEMP)
    return recortar(np.asarray(w[0], dtype=np.float32), sr), sr

partes, manifiesto, t, inicio = [], [], 0.0, time.time()
for i, seg in enumerate(segmentos):
    leido = normalizar(seg.texto, LEXICO)
    if ver_texto_leido: print(f"   leído: {leido}")
    esperado = max(1.0, len(leido) / 14.5)
    mejor = None
    for intento in range(3):  # reintenta si la duración sale anómala (repeticiones o cortes)
        s = semilla + i + intento * 7919
        audio, SR = generar(leido, s)
        ratio = len(audio) / SR / esperado
        if mejor is None or abs(math.log(max(ratio, 1e-3))) < abs(math.log(max(mejor[1], 1e-3))):
            mejor = (audio, ratio, s)
        if 0.55 <= ratio <= 1.8: break
    audio, ratio, s = mejor
    audio = fundidos(audio, SR)
    dur = len(audio) / SR
    ok = 0.55 <= ratio <= 1.8
    print(f"[{i+1}/{len(segmentos)}] {'✅' if ok else '⚠️ revisar'} {dur:5.1f}s  {seg.texto[:70]}…")
    manifiesto.append({"n": i + 1, "texto": seg.texto, "leido": leido, "inicio": round(t, 3), "fin": round(t + dur, 3),
                       "semilla": s, "ratio": round(ratio, 2), "ok": ok})
    pausa = pausa_entre_parrafos if seg.final_de_parrafo else pausa_entre_frases
    partes += [audio, np.zeros(int(pausa * SR), np.float32)]
    t += dur + pausa

final = sonoridad(np.concatenate(partes[:-1]), SR)
destino = CARPETA / "narraciones" / f"{nombre_archivo}_{time.strftime('%Y%m%d_%H%M%S')}"
sf.write(destino.with_suffix(".wav"), final, SR, subtype="PCM_16")
destino.with_suffix(".srt").write_text(srt(manifiesto), encoding="utf-8")
destino.with_suffix(".json").write_text(json.dumps({"voz": REF_WAV.name, "segmentos": manifiesto}, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"\n✅ {len(final)/SR:.1f} s de audio en {time.time()-inicio:.0f} s → {destino.with_suffix('.wav')}")
display(Audio(final, rate=SR))
from google.colab import files
files.download(str(destino.with_suffix(".wav"))); files.download(str(destino.with_suffix(".srt")))

## 5 · (Opcional) Pruebas de pronunciación
Genera frases difíciles con tu voz, las transcribe con Whisper y marca las que fallan. Lo que suene mal, añádelo al léxico de la celda «Cargar motor».

In [ ]:
#@title ▶️ Probar pronunciación
FRASES = [
 {
  "categoria": "Cifras",
  "texto": "Más de 6,4 millones de personas necesitan ayuda humanitaria en 2026."
 },
 {
  "categoria": "Cifras",
  "texto": "Entre enero y septiembre se documentaron 4.217 muertes y 1.860 heridos."
 },
 {
  "categoria": "Cifras",
  "texto": "Las bandas controlan entre el 80 y el 90 % de la capital."
 },
 {
  "categoria": "Cifras",
  "texto": "El plan de 880 millones de dólares solo alcanzó un 27 % de fondos."
 },
 {
  "categoria": "Fechas",
  "texto": "La dictadura de los Duvalier duró de 1957 a 1986."
 },
 {
  "categoria": "Fechas",
  "texto": "El 30 de septiembre de 2025 se aprobó la Resolución 2793."
 },
 {
  "categoria": "Siglas",
  "texto": "La ONU, la OIM, el BINUH y UNICEF publicaron nuevos informes."
 },
 {
  "categoria": "Siglas",
  "texto": "El análisis del IPC confirmó la crisis alimentaria."
 },
 {
  "categoria": "Nombres propios",
  "texto": "La emboscada ocurrió en Croix-des-Bouquets, cerca de Puerto Príncipe.",
  "revisar_oido": true
 },
 {
  "categoria": "Nombres propios",
  "texto": "El primer ministro Alix Didier Fils-Aimé encabeza el Ejecutivo.",
  "revisar_oido": true
 },
 {
  "categoria": "Nombres propios",
  "texto": "Coaliciones como Viv Ansanm dominan el departamento de Artibonite.",
  "revisar_oido": true
 },
 {
  "categoria": "Fonética",
  "texto": "El perro de Ramón corrió por la carretera junto al río."
 },
 {
  "categoria": "Fonética",
  "texto": "La cigüeña y el pingüino llegaron al año siguiente a la montaña."
 },
 {
  "categoria": "Fonética",
  "texto": "Él sabía que el té de la tía Lucía estaba frío."
 },
 {
  "categoria": "Entonación",
  "texto": "¿Cómo se llegó hasta aquí y qué está pasando hoy?"
 },
 {
  "categoria": "Entonación",
  "texto": "A pesar de todo, la cultura haitiana sigue viva."
 }
]
import base64, io, html, unicodedata, jiwer
from transformers import pipeline
from texto import normalizar_numeros
if "prompt" not in globals():
    liberar("diseno"); base = modelo("base")
    prompt = base.create_voice_clone_prompt(ref_audio=str(REF_WAV), ref_text=TEXTO_REF)
asr = pipeline("automatic-speech-recognition", model="openai/whisper-large-v3-turbo", torch_dtype=DTYPE, device="cuda:0")
def comparable(t):
    t = unicodedata.normalize("NFD", normalizar_numeros(t).lower())
    t = "".join(c for c in t if unicodedata.category(c) != "Mn")
    return re.sub(r"\s+", " ", re.sub(r"[^a-zñ0-9 ]+", " ", t)).strip()
filas, ok = [], 0
for i, f in enumerate(FRASES, 1):
    torch.manual_seed(semilla + i)
    w, sr = base.generate_voice_clone(text=normalizar(f["texto"], LEXICO), language="Spanish", voice_clone_prompt=prompt, temperature=TEMP)
    a = recortar(np.asarray(w[0], dtype=np.float32), sr)
    oido = asr({"raw": a, "sampling_rate": sr}, generate_kwargs={"language": "spanish"})["text"].strip()
    cer = jiwer.cer(comparable(f["texto"]), comparable(oido))
    estado = "revisar a oído" if f.get("revisar_oido") else ("OK" if cer <= 0.12 else "FALLA")
    ok += estado == "OK"
    buf = io.BytesIO(); sf.write(buf, a, sr, format="WAV"); b64 = base64.b64encode(buf.getvalue()).decode()
    color = {"OK": "#1f9d55", "FALLA": "#d64545", "revisar a oído": "#b7791f"}[estado]
    filas.append(f"<tr><td>{html.escape(f['texto'])}</td><td>{html.escape(oido)}</td><td>{cer:.2f}</td>"
                 f"<td style='color:{color};font-weight:700'>{estado}</td><td><audio controls src='data:audio/wav;base64,{b64}'></audio></td></tr>")
    print(f"{i:02d} {estado:14s} CER {cer:.2f}  {f['texto'][:60]}")
display(HTML(f"<p><b>{ok}/{len(FRASES)} OK</b></p><table style='border-collapse:collapse'><tr><th>Texto</th><th>Lo que entendió Whisper</th><th>CER</th><th>Estado</th><th>Audio</th></tr>{''.join(filas)}</table>"))